# [Demo] Tracing com Langfuse

Um log mostra que uma execução aconteceu e como terminou. Um trace mostra o caminho inteiro: cada chamada
ao modelo, cada chamada de tool, na ordem em que aconteceram.

O que vamos ver:
- instrumentando um agente com `CallbackHandler` do Langfuse;
- o trace de uma execução com tool call, contra o de uma sem;
- por que a mesma instrumentação serve tanto pra um Langfuse local quanto pra um Langfuse Cloud.

## Setup

Carregamos as libs, as variáveis de ambiente (`OPENAI_API_KEY` e as chaves `LANGFUSE_*`, todas no `.env`) e
inicializamos o modelo uma vez.

In [1]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI(model="gpt-4o-mini")

## Um agente com uma tool pra rastrear

O mesmo catálogo pequeno de horários por especialidade. Um trace de uma execução sem tool call mostra só
uma chamada ao modelo; com tool call, mostra as duas, em ordem. As duas ficam registradas mais abaixo, cada
uma com uma tag pra localizar na UI do Langfuse.

In [4]:
SLOTS = {
    "cardiologia": ["17/11/2026 09h", "17/11/2026 14h"],
    "dermatologia": ["18/11/2026 10h"],
}

In [5]:
@tool
def find_available_slots(specialty: str) -> str:
    """Consulta horários livres pra uma especialidade."""
    slots = SLOTS.get(specialty.lower())
    if not slots:
        return f"Não encontrei horários pra {specialty}."
    return ", ".join(slots)

In [6]:
PROMPT = (
    "Você é o assistente da Clínica Alura. A clínica atende cardiologia, dermatologia e clínica geral. "
    "Quando o pedido for sobre agendamento ou disponibilidade de horários, use a tool disponível pra "
    "consultar. Nunca invente uma informação que você não tem."
)

In [7]:
agent = create_agent(model=model, tools=[find_available_slots], system_prompt=PROMPT)

## Instrumentando com Langfuse

`CallbackHandler()` lê `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` e `LANGFUSE_HOST` do ambiente. Anexado
via `config={"callbacks": [handler]}`, cada execução do agente vira um trace. Uma tag em `metadata` deixa
esse trace fácil de achar entre todos os outros do mesmo projeto Langfuse.

In [9]:
handler = CallbackHandler()

In [10]:
response = agent.invoke(
    input={"messages": [HumanMessage("Quero agendar uma consulta de cardiologia")]},
    config={"callbacks": [handler], "metadata": {"langfuse_tags": ["com-tool-call"]}},
)
print(response["messages"][-1].content)

Os horários disponíveis para consulta de cardiologia são:

- 17 de novembro de 2026, às 09h
- 17 de novembro de 2026, às 14h

Qual desses horários você gostaria de agendar?


In [11]:
response

{'messages': [HumanMessage(content='Quero agendar uma consulta de cardiologia', additional_kwargs={}, response_metadata={}, id='3bd9443c-9b2d-45f3-815b-ef3701a6956e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 106, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_bb687c7345', 'id': 'chatcmpl-EMAWA6KWPD9UWMGaff1qyvw04ZZA5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a085de-9c20-7113-8fe3-599818780041-0', tool_calls=[{'name': 'find_available_slots', 'args': {'specialty': 'cardiologia'}, 'i

In [12]:
response = agent.invoke(
    input={"messages": [HumanMessage("Quais especialidades vocês atendem?")]},
    config={"callbacks": [handler], "metadata": {"langfuse_tags": ["sem-tool-call"]}},
)
print(response["messages"][-1].content)

Na Clínica Alura, atendemos nas seguintes especialidades: cardiologia, dermatologia e clínica geral.


In [13]:
response

{'messages': [HumanMessage(content='Quais especialidades vocês atendem?', additional_kwargs={}, response_metadata={}, id='3523167d-e7eb-4c81-8075-84101f48632a'),
  AIMessage(content='Na Clínica Alura, atendemos nas seguintes especialidades: cardiologia, dermatologia e clínica geral.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 105, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_bb687c7345', 'id': 'chatcmpl-EMAWQ9owIQcaXzagD4IzqeuIcP2KI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a085de-d919-72d2-ae98-57b1077770d2-0

In [14]:
get_client().flush()

O `flush` garante que os spans das duas execuções saem antes do processo encerrar. Sem
`LANGFUSE_PUBLIC_KEY`/`LANGFUSE_SECRET_KEY` configurados, o client do Langfuse fica desabilitado e não
envia nada, mas o agente roda normalmente: a instrumentação nunca trava a conversa.

## Local ou cloud, mesma instrumentação

`LANGFUSE_HOST` aponta pra um Langfuse local (Docker, `http://localhost:3000`) ou pra uma conta no Langfuse
Cloud. O código de instrumentação acima não muda: só as três variáveis no `.env` trocam.

## Takeaway

Um trace é a árvore de passos de uma execução específica: cada chamada ao modelo, cada chamada de tool, na
ordem em que aconteceram. `CallbackHandler` mais `config={"callbacks": [...]}` é toda a instrumentação
necessária; `flush` garante que os spans saem antes do processo terminar.